# 04c — UMAP Explained

## The One-Sentence Version
UMAP is like t-SNE but faster, with better global structure, and a topological foundation.

## What You'll Build Intuition For
- How UMAP differs from t-SNE conceptually
- Key parameters: `n_neighbors` and `min_dist`
- Speed advantages on larger datasets
- When to choose UMAP vs t-SNE

## Prerequisites
- [04b — t-SNE Explained](04b_tsne_explained.ipynb) — the method UMAP improves on

## The Story

t-SNE was a revelation when it appeared — finally, a way to see structure in high-dimensional data. But it had limitations: slow on large datasets, poor at preserving global structure, and sensitive to hyperparameters.

UMAP (Uniform Manifold Approximation and Projection) addresses all three. It's grounded in topology rather than probability, which gives it better mathematical properties. In practice, it produces embeddings where not only the local clusters are preserved, but the relative arrangement of clusters is more meaningful too.

UMAP has become the default nonlinear embedding method in most fields. Understanding how it differs from t-SNE helps you choose wisely.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import time
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import umap

from utils.plotting import apply_style, COLOURS, PALETTE
from utils.data_generators import make_digit_data

apply_style()
rng = np.random.default_rng(42)

## How UMAP Differs from t-SNE

| | t-SNE | UMAP |
|---|---|---|
| **Foundation** | Probability distributions | Fuzzy topological graphs |
| **Loss function** | KL divergence (asymmetric) | Cross-entropy (symmetric) |
| **Global structure** | Poor — inter-cluster distances meaningless | Better — relative cluster positions more meaningful |
| **Speed** | O(n log n) with Barnes-Hut | O(n) approximate — much faster |
| **Determinism** | Random initialisation | Random, but more stable |

The key practical difference: UMAP's symmetric loss function means it penalises both "nearby points pulled apart" and "distant points pushed together." t-SNE only penalises the first, which is why its global structure is less reliable.

In [ ]:
# UMAP vs t-SNE on digits
data, target, images = make_digit_data()

reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=42)
X_umap = reducer.fit_transform(data)

tsne = TSNE(n_components=2, perplexity=30, random_state=42)
X_tsne = tsne.fit_transform(data)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

for digit in range(10):
    mask = target == digit
    ax1.scatter(X_tsne[mask, 0], X_tsne[mask, 1], s=10, alpha=0.5,
               label=str(digit), color=PALETTE[digit % len(PALETTE)])
    ax2.scatter(X_umap[mask, 0], X_umap[mask, 1], s=10, alpha=0.5,
               label=str(digit), color=PALETTE[digit % len(PALETTE)])

ax1.set_title("t-SNE")
ax1.set_xticks([])
ax1.set_yticks([])

ax2.set_title("UMAP")
ax2.set_xticks([])
ax2.set_yticks([])
ax2.legend(title="Digit", markerscale=3, fontsize=8, loc="upper right")

fig.suptitle("t-SNE vs UMAP on digits — both find clusters, UMAP preserves more global structure",
             fontweight="bold")
plt.tight_layout()
plt.savefig("visuals/04c_tsne_vs_umap.png", dpi=150, bbox_inches="tight")
plt.show()

print("Both methods find the 10 digit clusters clearly.")
print("UMAP often places similar digits (e.g. 4 and 9) closer together — better global layout.")

## Key Parameters

UMAP has two main knobs:

- **`n_neighbors`** (like perplexity): how many neighbours define "local." Low values = fine local detail. High values = broader, smoother embedding.
- **`min_dist`**: how tightly packed the embedding is. Low values = tight clumps. High values = more spread out, preserving broader topology.

In [ ]:
# Grid: n_neighbors × min_dist
nn_values = [5, 15, 50]
md_values = [0.0, 0.25, 0.8]

fig, axes = plt.subplots(len(nn_values), len(md_values), figsize=(14, 14))

for row, nn in enumerate(nn_values):
    for col, md in enumerate(md_values):
        reducer = umap.UMAP(n_components=2, n_neighbors=nn, min_dist=md,
                            random_state=42)
        X_emb = reducer.fit_transform(data)

        ax = axes[row, col]
        for digit in range(10):
            mask = target == digit
            ax.scatter(X_emb[mask, 0], X_emb[mask, 1], s=5, alpha=0.4,
                       color=PALETTE[digit % len(PALETTE)])

        ax.set_xticks([])
        ax.set_yticks([])
        if row == 0:
            ax.set_title(f"min_dist={md}", fontsize=11)
        if col == 0:
            ax.set_ylabel(f"n_neighbors={nn}", fontsize=11)

fig.suptitle("UMAP parameter grid: n_neighbors × min_dist", fontweight="bold", fontsize=14)
plt.tight_layout()
plt.savefig("visuals/04c_umap_params.png", dpi=150, bbox_inches="tight")
plt.show()

print("Top-left (few neighbours, tight): very fine-grained, fragmented clusters.")
print("Bottom-right (many neighbours, spread): smooth, broad structure.")
print("Middle row, middle column: the sweet spot for most data.")

## Speed Comparison

On larger datasets, UMAP's speed advantage is dramatic. Let's time both methods on increasing subsets of the digits data.

In [ ]:
# Speed comparison: t-SNE vs UMAP on increasing data sizes
# Use digits data (1797 samples) and subsample
sizes = [200, 500, 1000, 1797]
tsne_times = []
umap_times = []

for n in sizes:
    X_sub = data[:n]

    start = time.time()
    TSNE(n_components=2, perplexity=min(30, n//4), random_state=42).fit_transform(X_sub)
    tsne_times.append(time.time() - start)

    start = time.time()
    umap.UMAP(n_components=2, n_neighbors=15, random_state=42).fit_transform(X_sub)
    umap_times.append(time.time() - start)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(sizes, tsne_times, "o-", color=COLOURS["noise"], linewidth=2,
        markersize=8, label="t-SNE")
ax.plot(sizes, umap_times, "o-", color=COLOURS["signal"], linewidth=2,
        markersize=8, label="UMAP")
ax.set_xlabel("Number of Samples")
ax.set_ylabel("Time (seconds)")
ax.set_title("t-SNE vs UMAP: execution time")
ax.legend()
plt.tight_layout()
plt.savefig("visuals/04c_speed_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

print("Timing results:")
for n, tt, ut in zip(sizes, tsne_times, umap_times):
    speedup = tt / ut if ut > 0 else float('inf')
    print(f"  n={n:>5}: t-SNE {tt:.2f}s, UMAP {ut:.2f}s ({speedup:.1f}× faster)")

print("\nUMAP's advantage grows with dataset size.")
print("On 100k+ points, t-SNE becomes impractical while UMAP remains fast.")

## When to Choose Which

| Situation | Recommendation |
|-----------|---------------|
| Default visualisation of any dataset | **UMAP** — faster, better global structure |
| Need very precise local structure | **t-SNE** — slightly better at fine local detail |
| Large dataset (>10k points) | **UMAP** — t-SNE becomes too slow |
| Need reproducible layout | **UMAP** — more stable across runs |
| Downstream ML features | **Neither** — use PCA or autoencoders |

> **Rule of thumb:** Start with UMAP. Only switch to t-SNE if you need finer local structure and have a small enough dataset.

<details>
<summary><b>The Maths Behind This</b></summary>

UMAP constructs a **fuzzy simplicial set** (weighted graph) in both high-D and low-D:

**High-D graph:** For each point $i$, find $k$ nearest neighbours. Assign edge weights using:

$$w_{ij} = \exp\left(-\frac{d(\mathbf{x}_i, \mathbf{x}_j) - \rho_i}{\sigma_i}\right)$$

where $\rho_i$ is the distance to the nearest neighbour and $\sigma_i$ is calibrated to match the desired `n_neighbors`. Symmetrise: $w_{ij} = w_{ij} + w_{ji} - w_{ij} \cdot w_{ji}$.

**Low-D graph:** Edge weights using:

$$v_{ij} = \left(1 + a \|\mathbf{y}_i - \mathbf{y}_j\|^{2b}\right)^{-1}$$

where $a, b$ are derived from `min_dist`.

**Loss:** Binary cross-entropy between the two graphs:

$$\mathcal{L} = \sum_{ij} \left[ w_{ij} \log\frac{w_{ij}}{v_{ij}} + (1 - w_{ij}) \log\frac{1 - w_{ij}}{1 - v_{ij}} \right]$$

Unlike t-SNE's KL divergence, this is symmetric — it penalises both missing neighbours and false neighbours, which is why UMAP preserves more global structure.

</details>

## Where This Matters

**Single-cell biology:** UMAP has largely replaced t-SNE as the standard visualisation. Tools like Scanpy and Seurat use UMAP by default for cell type discovery in scRNA-seq data with 10,000–100,000+ cells.

**Cybersecurity:** Network traffic analysis involves millions of flow records with dozens of features. UMAP can embed these in 2D for visual anomaly detection — finding attack patterns that statistical methods miss.

## Feynman Check

1. **What does UMAP preserve that t-SNE doesn't?**

2. **Which would you use on a dataset with 100,000 points? Why?**

Now let's put everything side by side in **04d — Comparison Arena**, where we run all methods on the same datasets and see what each reveals.